# Exploratory Data Analysis (EDA) and Physics Baseline

This notebook performs exploratory data analysis on the Khavda 5-year historical dataset. It focuses on the primary features driving wind power generation forecasting, and outlines the physical baseline calculations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Prettify plots
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Data Ingestion
We load the historical wind data.

In [ ]:
df = pd.read_csv('../data/01_raw/khavda_5yr_historical.csv')
df['time'] = pd.to_datetime(df['time'])
df.set_index('time', inplace=True)
display(df.head())

## 2. Basic Statistics & Missing Values

In [ ]:
display(df.info())
display(df.describe())

## 3. Univariate Analysis
Let's look at the distribution of the Key features: `wind_speed_100m`, `temperature_2m`, and `wind_gust`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df['wind_speed_100m'], bins=50, kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Wind Speed (100m) Distribution')

sns.histplot(df['temperature_2m'], bins=50, kde=True, ax=axes[1], color='salmon')
axes[1].set_title('Temperature (2m) Distribution')

sns.histplot(df['wind_gust'], bins=50, kde=True, ax=axes[2], color='lightgreen')
axes[2].set_title('Wind Gust Distribution')

plt.tight_layout()
plt.show()

## 4. Correlation Analysis
Checking how weather features correlate with each other.

In [ ]:
plt.figure(figsize=(10, 8))
corr = df.corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.show()

## 5. Time Series Trends
We resample the data to daily and monthly averages to observe seasonality.

In [ ]:
monthly_avg = df['wind_speed_100m'].resample('ME').mean()

plt.figure(figsize=(14, 5))
sns.lineplot(x=monthly_avg.index, y=monthly_avg.values, marker='o', color='teal')
plt.title('Monthly Average Wind Speed (100m)')
plt.xlabel('Date')
plt.ylabel('Wind Speed (m/s)')
plt.show()

## 6. Wind Rose (Direction & Speed)
Wind roses are essential to understand the dominant directions of wind power.

In [ ]:
# Note: Need windrose lib, else we use a polar generic plot
import matplotlib.cm as cm

plt.figure(figsize=(8, 8))
ax = plt.subplot(111, polar=True)
ax.set_theta_direction(-1)
ax.set_theta_zero_location('N')

dir_rad = np.radians(df['wind_direction_100m'])
ax.scatter(dir_rad, df['wind_speed_100m'], alpha=0.1, s=1, c='blue')
plt.title('Wind Rose (Direction vs Speed)')
plt.show()

## 7. Physics Baseline Calculation
Next, we compute a baseline physical measure, defining theoretical power using simplified air density.

In [ ]:
# Gas constant for dry air in J/(kg·K)
R_specific = 287.058

# Calculate Air Density (rho) -> p / (R * T)
# p = surface_pressure in Pa (1 hPa = 100 Pa)
# T = temperature_2m in Kelvin (+273.15)

df['pressure_pa'] = df['surface_pressure'] * 100
df['temperature_k'] = df['temperature_2m'] + 273.15
df['air_density'] = df['pressure_pa'] / (R_specific * df['temperature_k'])

# Calculate theoretical power for a turbine: P = 0.5 * rho * A * v^3 * Cp
# We will calculate Power Flux Density = 0.5 * rho * v^3 (W/m^2)
df['power_flux_density'] = 0.5 * df['air_density'] * (df['wind_speed_100m'] ** 3)

plt.figure(figsize=(14, 5))
df['power_flux_density'].resample('ME').mean().plot(color='purple', marker='o')
plt.title('Monthly Average Theoretical Power Flux Density (W/m²)')
plt.ylabel('Power Flux Density (W/m²)')
plt.show()